# 00 · Set up your lab and understand the moving parts

**Level:** beginner · **Time:** 30–45 minutes, plus first-time downloads

**You will:** start RabbitMQ, select the right Python kernel, inspect the broker, and explain
producer, exchange, queue, consumer and acknowledgement. You need basic Python variables,
dictionaries and functions. No RabbitMQ knowledge is assumed.

This is a practical course: read, predict, run, observe, then change something. The numbered
notebooks progress to Django/Celery, outbox delivery and quorum failover. Run one notebook at
a time, from top to bottom. Use **Restart Kernel and Run All** when repeating a lesson.


## 1. Start the local services

Install Python 3.12 or 3.13 and Docker Desktop with Linux containers. Open a terminal in the
repository directory. On Windows PowerShell, run:

```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.lock
Copy-Item .env.example .env
docker compose up -d --wait rabbitmq
.\.venv\Scripts\python.exe -m ipykernel install --sys-prefix --name python3 --display-name "RabbitMQ course"
.\.venv\Scripts\python.exe -m jupyterlab
```

Reuse an existing `.venv` and `.env` instead of overwriting custom settings. On macOS/Linux use
`.venv/bin/python` and `cp .env.example .env`. Full instructions are in `README.md`.

Choose **RabbitMQ course** in Jupyter's kernel selector. The kernel is the Python process that
executes these cells; installing a package in a different Python does not install it here.

Before lessons 12–15 and 18, run `docker compose up -d --build --wait` and check
`docker compose logs --tail 30 worker` for `ready`. This also starts PostgreSQL, Django and
the Linux Celery worker. Standalone messaging lessons need only RabbitMQ.


## 2. Check which interpreter this notebook uses

**Predict:** the executable path contains `.venv`, and every package below has a version.
If any import/package is missing, select the course kernel before continuing.


In [ ]:
from pathlib import Path
import sys
import importlib.metadata

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Course directory:", ROOT)
assert (3, 12) <= sys.version_info[:2] <= (3, 13), "Use Python 3.12 or 3.13 for the supplied locks."
for package in ["pika", "Django", "celery", "aio-pika", "jupyterlab"]:
    print(package, importlib.metadata.version(package))


## 3. Identify the services

| Service | Address from your notebook | Purpose |
| --- | --- | --- |
| RabbitMQ AMQP | `localhost:5672` | Send and receive messages |
| RabbitMQ Management | `http://localhost:15672` | Inspect queues and metrics |
| Django (later lessons) | `http://localhost:8000` | Create and inspect orders |
| PostgreSQL | Internal Compose network | Store Django business data |

The container hostname `rabbitmq` is used by the Django and worker containers. Your host-side
notebook uses `localhost`. The virtual host `learning` is a logical namespace within RabbitMQ.
Local teaching credentials are `student` / `student-local-only`.

The next cell checks TCP ports; a reachable port alone does not prove AMQP authentication works.
Django can be unavailable here if you started only the broker.


In [ ]:
import shutil
import socket

print("Docker CLI:", shutil.which("docker") or "Not found: install Docker Desktop and reopen your terminal")
for port, service in [(5672, "AMQP"), (15672, "Management"), (8000, "Django, needed later")]:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            print(f"{service}: reachable on {port}")
    except OSError:
        print(f"{service}: unavailable on {port}")


## 4. Learn the message path

```text
Producer ── publish(exchange, routing key, body) ──> Exchange
                                                      │ binding rule
                                                      ▼
                                                    Queue
                                                      │ delivery
                                                      ▼
                                                   Consumer
                                                      │ acknowledgement
                                                      └─────> RabbitMQ
```

- **Producer:** application sending work or events.
- **Exchange:** routes each publication to zero or more queues using bindings.
- **Queue:** holds messages for consumers; a queue is not a Python list in your program.
- **Consumer:** receives messages and performs application work.
- **Acknowledgement:** consumer tells RabbitMQ that a delivery has been handled.
- **Publisher confirm:** broker tells a publisher it accepted responsibility for a publication.

A confirm does not prove a consumer finished. RabbitMQ delivers message bodies as bytes; JSON is
an application serialization choice. Two consumers on one queue share deliveries; two queues
bound to an exchange can each get a copy. We will demonstrate both behaviors.


## 5. Build a message before sending anything

An event ID identifies the logical event even if it is delivered again. A schema version lets a
consumer distinguish payload formats. The timestamp describes when the event was created.
**Predict:** serializing then parsing this JSON preserves all the values.


In [ ]:
import json
import uuid
from datetime import datetime, timezone

event = {
    "schema_version": 1,
    "event_id": str(uuid.uuid4()),
    "event_type": "order.created",
    "occurred_at": datetime.now(timezone.utc).isoformat(),
    "data": {"order_id": 101, "product": "Learning RabbitMQ", "quantity": 2},
}
wire_body = json.dumps(event).encode("utf-8")
assert isinstance(wire_body, bytes)
assert json.loads(wire_body) == event
print(json.dumps(event, indent=2))
print("Wire bytes:", len(wire_body))


## 6. Authenticate and declare a temporary queue

`course/rabbit.py` loads connection settings from `.env` and bounds connection waits. It does
not hide publishing or acknowledging; those Pika operations remain visible in the lessons.
This first check creates an **exclusive** queue with a broker-generated name. It is removed
when this connection closes. It publishes no message.

**Expected:** a queue name appears and its initial message count is zero. Connection refusal
means the service is not ready; access refusal means credentials or vhost permissions need checking.


In [ ]:
from course.rabbit import connect

try:
    with connect() as connection:
        channel = connection.channel()
        declared = channel.queue_declare(queue="", exclusive=True)
        assert declared.method.message_count == 0
        print("Authenticated. Temporary queue:", declared.method.queue)
except Exception as error:
    raise RuntimeError(
        "RabbitMQ connection check failed. Start Docker Desktop, run "
        "'docker compose up -d --wait rabbitmq', and check .env credentials/vhost. "
        f"Underlying exception: {type(error).__name__}"
    ) from error
print("Connection closed; temporary queue removed.")


## 7. Inspect Management without exposing connection secrets

Open http://localhost:15672 and sign in. Select the `learning` virtual host. The HTTP API below
checks the same broker and reports its version. Management statistics are sampled and may lag.
Do not infer successful business processing from a dashboard count alone.


In [ ]:
import os
import requests

management_url = os.getenv("RABBITMQ_MANAGEMENT_URL", "http://localhost:15672").rstrip("/")
session = requests.Session()
session.auth = (os.getenv("RABBITMQ_USER", "student"), os.getenv("RABBITMQ_PASSWORD", "student-local-only"))
try:
    response = session.get(f"{management_url}/api/overview", timeout=5)
    response.raise_for_status()
    overview = response.json()
    print("RabbitMQ version:", overview["rabbitmq_version"])
    response = session.get(f"{management_url}/api/vhosts", timeout=5)
    response.raise_for_status()
    print("Visible virtual hosts:", [item["name"] for item in response.json()])
finally:
    session.close()


## 8. Exercise: predict where work waits

For each scenario, make a prediction before reading the suggested answer.

1. A producer publishes to an existing queue while no consumer runs. Where is the work?
2. Two consumers listen to one queue. Does each get every message?
3. Two queues are bound to a fanout exchange. How many copies can one publication create?
4. A consumer changes the database, then crashes before acknowledging. Can the event arrive again?

<details><summary>Suggested answers</summary>

1. It can wait in the queue, subject to policies such as expiry and size limits.
2. Deliveries are distributed among consumers; redelivery can later reach another consumer.
3. One routed copy in each bound queue, so two here.
4. Yes. Use an idempotent business operation; acknowledging before the change would introduce a loss window.

</details>


## Troubleshooting and completion check

- If the environment check cannot find the repository, launch Jupyter from this repository or `notebooks/`.
- If `.env` changes after imports, restart the kernel; imported settings may still be cached.
- AMQP uses 5672, not the Management HTTP port 15672.
- Container defaults initialize users/vhosts on a fresh volume; changing them does not rewrite an existing volume.
- Keep Docker running while practicing. `docker compose stop` stops services and preserves data.

You are ready when authentication and Management checks pass and you can name each step of the
message path. Continue with **01 · Messaging basics**.

References: [RabbitMQ Python tutorials](https://www.rabbitmq.com/tutorials),
[acknowledgements and confirms](https://www.rabbitmq.com/docs/confirms),
[Management plugin](https://www.rabbitmq.com/docs/management).
